## Web Datamining & Semantics 
## Lab 4 : Knowledge Base Construction, Alignment, and Expansion
This notebook implements a full pipeline to build a **private Knowledge Base** from web-mined data,
align it with Wikidata, expand it via SPARQL queries, and prepare it for **Knowledge Graph Embedding**.

### Pipeline Overview
| Step | Description | Output |
|------|-------------|--------|
| 1 | Build RDF graph from CSV files | `private_kb.nt` |
| 2 | Entity linking with Wikidata | `alignment_entities.ttl` |
| 3 | Predicate alignment via SPARQL | `alignment_predicates.ttl` |
| 4 | KB expansion via SPARQL queries | `expanded_kb.nt` |
| 5 | Cleaning & KGE preparation | `kge/train.txt`, `valid.txt`, `test.txt` |
| 6 | Final deliverables & statistics | `stats_report.txt` |

### Input Data
- `extracted_knowledge.xls` : 468 named entities (PERSON, ORG, GPE, DATE) extracted from [distill.pub](https://distill.pub)
- `extracted_relations.xls` :  523 co-occurrence relations between entities
  
### Step 1 : Build the Initial Private Knowledge Base

#### Objective
Transform the raw CSV files into a structured **RDF graph** using `rdflib`.


#### NER Type Mapping
Each entity type from spaCy is mapped to an OWL class:
```
PERSON → onto:Person      (rdfs:subClassOf schema:Person)
ORG    → onto:Organization (rdfs:subClassOf schema:Organization)
GPE    → onto:GeopoliticalEntity
DATE   → onto:Date
```

In [6]:
import csv
import re
import unicodedata
from pathlib import Path
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

In [4]:
# Namespaces 
LOCAL = Namespace("http://ai-kb.local/entity/")
ONTO = Namespace("http://ai-kb.local/ontology/")
SCHEMA = Namespace("https://schema.org/")
PROV = Namespace("http://www.w3.org/ns/prov#")

In [5]:
#Paths of files
INPUT_ENTITIES = "../data/extracted_knowledge.csv"
INPUT_RELATIONS = "../data/extracted_relations.csv"
OUTPUT_DIR = Path("output")
OUTPUT_NT = OUTPUT_DIR / "private_kb.nt"
OUTPUT_DIR.mkdir(exist_ok=True)

In [6]:
def slugify(text: str) -> str:
    """
    Transforms a string into a valid slug for a URI.
    
    """
    # Unicode normalization 
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    # Remove characters not allowed in URIs
    text = re.sub(r"[^\w\s-]", "", text)
    # CamelCase: capitalize each word and remove spaces
    text = "".join(w.capitalize() for w in text.split())
    return text if text else "Unknown"

In [7]:
def make_entity_uri(name: str) -> URIRef:
    return LOCAL[slugify(name)]

In [8]:
def make_predicate_uri(pred: str) -> URIRef:
    """
    Converts a CSV predicate into a camelCase URI.
    
    """
    parts = pred.strip().split("_")
    camel = parts[0].lower() + "".join(p.capitalize() for p in parts[1:])
    return ONTO[camel]

In [9]:
NER_TYPE_MAP = {
    "PERSON": ONTO.Person,
    "ORG":    ONTO.Organization,
    "GPE":    ONTO.GeopoliticalEntity,
    "DATE":   ONTO.Date,
    "LOC":    ONTO.Location,
}

In [10]:
# Graph construction 
def build_graph(entities_file: Path, relations_file: Path) -> Graph:
    g = Graph()

    # Namespace declarations 
    g.bind("local",  LOCAL)
    g.bind("onto",   ONTO)
    g.bind("schema", SCHEMA)
    g.bind("owl",    OWL)
    g.bind("rdfs",   RDFS)
    g.bind("prov",   PROV)

    # 1. Minimal ontology (classes + properties) 
    classes = [
        (ONTO.Person,             "Person",             "A human individual"),
        (ONTO.Organization,       "Organization",       "A company, university, or research lab"),
        (ONTO.GeopoliticalEntity, "GeopoliticalEntity", "A country, city, or region"),
        (ONTO.Date,               "Date",               "A temporal reference"),
        (ONTO.Location,           "Location",           "A geographical location"),
    ]
    for cls_uri, label, comment in classes:
        g.add((cls_uri, RDF.type,       OWL.Class))
        g.add((cls_uri, RDFS.label,     Literal(label, lang="en")))
        g.add((cls_uri, RDFS.comment,   Literal(comment, lang="en")))

    # Semantic subclasses
    g.add((ONTO.Person,             RDFS.subClassOf, SCHEMA.Person))
    g.add((ONTO.Organization,       RDFS.subClassOf, SCHEMA.Organization))
    g.add((ONTO.GeopoliticalEntity, RDFS.subClassOf, SCHEMA.Place))

    # Base properties
    props = [
        (ONTO.relatedTo,    "relatedTo",    "Generic co-occurrence relation between entities"),
        (ONTO.extractedFrom,"extractedFrom","Source URL from which this entity was extracted"),
        (ONTO.entityLabel,  "entityLabel",  "Raw surface form of the entity as extracted"),
    ]
    for prop_uri, label, comment in props:
        g.add((prop_uri, RDF.type,       OWL.ObjectProperty))
        g.add((prop_uri, RDFS.label,     Literal(label, lang="en")))
        g.add((prop_uri, RDFS.comment,   Literal(comment, lang="en")))

    # extractedFrom is a DatatypeProperty 
    g.add((ONTO.extractedFrom, RDF.type, OWL.DatatypeProperty))
    g.add((ONTO.extractedFrom, RDFS.range, XSD.anyURI))

    # 2. Entities 
    entities_added = set()
    print(f"\n[STEP 1] Reading {entities_file}...")

    with open(entities_file, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            raw_name = row.get("entity", "").strip()
            ner_type  = row.get("entity_type", "").strip()
            source    = row.get("source_url", "").strip()

            if not raw_name or ner_type not in NER_TYPE_MAP:
                continue  # Skip invalid rows or unwanted NER types

            uri = make_entity_uri(raw_name)

            if uri not in entities_added:
                # RDF type
                g.add((uri, RDF.type,       NER_TYPE_MAP[ner_type]))
                # Human-readable label
                g.add((uri, RDFS.label,     Literal(raw_name, lang="en")))
                # Original raw form (useful for debugging)
                g.add((uri, ONTO.entityLabel, Literal(raw_name)))
                entities_added.add(uri)

            # Source 
            if source:
                g.add((uri, ONTO.extractedFrom, Literal(source, datatype=XSD.anyURI)))

    print(f"  ✓ {len(entities_added)} unique entities added")

    # 3. Relations 
    relations_added = 0
    skipped = 0
    print(f"\n[STEP 1] Reading {relations_file}...")

    with open(relations_file, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            subj_raw = row.get("subject", "").strip()
            pred_raw = row.get("relation", "").strip()
            obj_raw  = row.get("object",  "").strip()
            source   = row.get("source_url", "").strip()

            if not subj_raw or not pred_raw or not obj_raw:
                skipped += 1
                continue

            subj_uri = make_entity_uri(subj_raw)
            pred_uri = make_predicate_uri(pred_raw)
            obj_uri  = make_entity_uri(obj_raw)

            # Main triple
            g.add((subj_uri, pred_uri, obj_uri))

            # If subject/object are not already declared as entities :  add minimal declaration
            if subj_uri not in entities_added:
                g.add((subj_uri, RDF.type,   ONTO.Entity))
                g.add((subj_uri, RDFS.label, Literal(subj_raw, lang="en")))
                entities_added.add(subj_uri)

            if obj_uri not in entities_added:
                g.add((obj_uri, RDF.type,   ONTO.Entity))
                g.add((obj_uri, RDFS.label, Literal(obj_raw, lang="en")))
                entities_added.add(obj_uri)

            # Relation provenance
            if source:
                g.add((subj_uri, ONTO.extractedFrom, Literal(source, datatype=XSD.anyURI)))

            relations_added += 1

    print(f"   {relations_added} relations added ({skipped} rows skipped)")

    return g

In [11]:
def export_graph(g: Graph):
    # Export N-Triples (the most compatible format with KGE tools)
    g.serialize(destination=str(OUTPUT_NT), format="nt")
    print(f"\n[STEP 1] KB exported → {OUTPUT_NT}")

    # Export Turtle for human readability
    ttl_path = OUTPUT_DIR / "private_kb.ttl"
    g.serialize(destination=str(ttl_path), format="turtle")
    print(f"[STEP 1] Human-readable KB → {ttl_path}")

    # Statistics
    total   = len(g)
    entities = len(set(g.subjects(RDF.type, None)))
    preds    = len(set(g.predicates()))
    print(f"\n{'='*50}")
    print(f"  Total triples     : {total:>8,}")
    print(f"  Entities          : {entities:>8,}")
    print(f"  Unique predicates : {preds:>8,}")
    print(f"{'='*50}")

    return total, entities, preds

In [12]:
if __name__ == "__main__":
    g = build_graph(INPUT_ENTITIES, INPUT_RELATIONS)
    export_graph(g)



[STEP 1] Reading ../data/extracted_knowledge.csv...
  ✓ 398 unique entities added

[STEP 1] Reading ../data/extracted_relations.csv...
   523 relations added (0 rows skipped)

[STEP 1] KB exported → output\private_kb.nt


C:\Users\sagro\anaconda3\envs\tp_wine\lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


[STEP 1] Human-readable KB → output\private_kb.ttl

  Total triples     :    2,246
  Entities          :      426
  Unique predicates :        8


### Step 1 : Results

The initial KB was built from the two CSV files:

- **2,246 triples** generated
- **434 unique entities** (297 persons, 74 organizations, 21 dates, 6 GPE)
- **8 predicates** 

The graph is exported in two formats:
- `output/private_kb.nt`
- `output/private_kb.ttl` 

The `related_to` predicate is generic, it captures co-occurrence between entities
in the same sentence or paragraph. It will be semantically enriched in Step 3 via predicate alignment.

### Step 2 : Entity Linking with Open Knowledge Bases

#### Objective
Link each local entity to its corresponding Wikidata item using `owl:sameAs`.

#### Method
For each entity, we query the Wikidata Search API (`wbsearchentities`)

A **confidence score** is computed using Jaccard similarity between the local label
and the Wikidata label, with a +0.1 bonus if the description matches the expected NER type.

#### Decision Rule
| Score | Action |
|-------|--------|
| ≥ 0.75 | `owl:sameAs` → Wikidata URI |
| < 0.75 | Declared as `owl:NamedIndividual` with local ontological definition |


In [20]:
import json
import time
import requests

In [21]:
CONFIDENCE_THRESHOLD = 0.75   # Threshold to accept a Wikidata match
DELAY_BETWEEN_CALLS  = 1.0    # Seconds between API calls 
MAX_RETRIES = 5  # Maximum number of retries in case of 429
USE_CACHE = True   # Save API results to avoid repeated queries
MAX_ENTITIES = None   # None = all


In [22]:
#Namespaces
LOCAL = Namespace("http://ai-kb.local/entity/")
ONTO  = Namespace("http://ai-kb.local/ontology/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

In [23]:
# NER type : expected Wikidata types
# Used to boost the score if the type matches
NER_TO_WIKIDATA_TYPES = {
    "Person":             ["Q5"],            
    "Organization":       ["Q43229","Q7210356","Q8500405"],  
    "GeopoliticalEntity": ["Q6256","Q515","Q35657"],        
}

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_FILE = OUTPUT_DIR / "wikidata_cache.json"

In [24]:
def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^\w\s-]", "", text)
    return "".join(w.capitalize() for w in text.split())

In [25]:
def load_cache():
    if USE_CACHE and CACHE_FILE.exists():
        with open(CACHE_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_cache(cache: dict):
    if USE_CACHE:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)

In [26]:
def string_similarity(a: str, b: str) -> float:
    """
    Simple similarity score based on common tokens.
    
    """
    a_tokens = set(a.lower().split())
    b_tokens = set(b.lower().split())
    if not a_tokens or not b_tokens:
        return 0.0
    intersection = a_tokens & b_tokens
    union = a_tokens | b_tokens
    return len(intersection) / len(union)

In [27]:
def search_wikidata(entity_name: str, ner_type: str, cache: dict) -> list[dict]:
    """
    Searches for an entity on Wikidata via the text search API.

    """
    cache_key = f"{entity_name}||{ner_type}"
    if cache_key in cache:
        return cache[cache_key]

    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action":   "wbsearchentities",
        "search":   entity_name,
        "language": "en",
        "type":     "item",
        "limit":    5,
        "format":   "json",
    }
    headers = {
        "User-Agent": "AI-KB-Builder/1.0 (educational project; contact: student@university.edu)",
        "Accept":     "application/json",
    }

    data = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=15)

            if resp.status_code == 429:
                # Rate limit: exponential backoff
                wait = 2 ** (attempt + 1)
                print(f"  [429] Rate limit for '{entity_name}' → waiting {wait}s (attempt {attempt+1}/{MAX_RETRIES})")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            data = resp.json()
            break  # Success → exit loop

        except requests.exceptions.Timeout:
            wait = 2 ** (attempt + 1)
            print(f"  [TIMEOUT] '{entity_name}' → waiting {wait}s")
            time.sleep(wait)

        except Exception as e:
            print(f"  [WARNING] API error for '{entity_name}': {e}")
            break

    if data is None:
        # All attempts failed → cache empty list to avoid retrying
        cache[cache_key] = []
        time.sleep(DELAY_BETWEEN_CALLS)
        return []

    candidates = []
    for item in data.get("search", []):
        qid   = item.get("id", "")
        label = item.get("label", "")
        desc  = item.get("description", "")

        # Base score: textual similarity
        base_score = string_similarity(entity_name, label)

        # Bonus if description mentions expected type
        type_keywords = {
            "Person":             ["researcher", "scientist", "engineer", "professor", "human", "computer"],
            "Organization":       ["company", "organization", "university", "institute", "lab", "research"],
            "GeopoliticalEntity": ["country", "city", "state", "region", "nation"],
        }
        bonus = 0.0
        for kw in type_keywords.get(ner_type, []):
            if kw in desc.lower():
                bonus = 0.1
                break

        final_score = min(base_score + bonus, 1.0)

        candidates.append({
            "qid":         qid,
            "label":       label,
            "description": desc,
            "score":       round(final_score, 3),
            "uri":         f"http://www.wikidata.org/entity/{qid}",
        })

    # Sort by descending score
    candidates.sort(key=lambda x: x["score"], reverse=True)
    cache[cache_key] = candidates

    # Polite delay between requests
    time.sleep(DELAY_BETWEEN_CALLS)
    return candidates

In [28]:
#Load entities from RDF KB
def load_entities_from_graph(kb_path: Path) -> list[tuple[str, str, URIRef]]:
    """
    Loads entities and their types from the RDF KB generated in step 1.
    """
    g = Graph()
    g.parse(str(kb_path), format="nt")

    ONTO_local = Namespace("http://ai-kb.local/ontology/")
    type_map = {
        str(ONTO_local.Person):             "Person",
        str(ONTO_local.Organization):       "Organization",
        str(ONTO_local.GeopoliticalEntity): "GeopoliticalEntity",
        str(ONTO_local.Date):               "Date",
    }

    entities = []
    seen = set()
    for subj, _, rdf_type in g.triples((None, RDF.type, None)):
        type_str = type_map.get(str(rdf_type))
        if type_str and str(subj) not in seen:
            label_triple = list(g.triples((subj, RDFS.label, None)))
            label = str(label_triple[0][2]) if label_triple else str(subj).split("/")[-1]
            if type_str != "Date":  # Do not link dates to Wikidata
                entities.append((label, type_str, subj))
                seen.add(str(subj))

    return entities


In [29]:
#Build output graphs
def run_entity_linking(kb_path: Path):
    print(f"\n[STEP 2] Loading KB: {kb_path}")
    entities = load_entities_from_graph(kb_path)
    print(f"  ✓ {len(entities)} entities to link")

    if MAX_ENTITIES:
        entities = entities[:MAX_ENTITIES]
        print(f"  [INFO] Test mode: limited to {MAX_ENTITIES} entities")

    cache = load_cache()

    # Clear failed cache entries
    failed_keys = [k for k, v in cache.items() if v == []]
    if failed_keys:
        print(f"  [INFO] Clearing {len(failed_keys)} failed cache entries (retrying)")
        for k in failed_keys:
            del cache[k]

    # Output graphs
    g_align = Graph()  # owl:sameAs
    g_new   = Graph()  # new entities

    g_align.bind("local", LOCAL)
    g_align.bind("wd",    WD)
    g_align.bind("owl",   OWL)

    g_new.bind("local", LOCAL)
    g_new.bind("onto",  ONTO)
    g_new.bind("owl",   OWL)

    mapping_rows = []  # for CSV report

    matched   = 0
    unmatched = 0

    for i, (label, ner_type, local_uri) in enumerate(entities):
        if (i + 1) % 20 == 0:
            print(f"  ... {i+1}/{len(entities)} entities processed")
            save_cache(cache)

        candidates = search_wikidata(label, ner_type, cache)
        best = candidates[0] if candidates else None

        if best and best["score"] >= CONFIDENCE_THRESHOLD:
            # Match found → owl:sameAs 
            wd_uri = URIRef(best["uri"])
            g_align.add((local_uri, OWL.sameAs, wd_uri))

            # add Wikidata label as additional rdfs:label
            if best["label"] != label:
                g_align.add((local_uri, RDFS.label, Literal(best["label"], lang="en")))

            mapping_rows.append({
                "private_entity":  str(local_uri),
                "entity_label":    label,
                "ner_type":        ner_type,
                "wikidata_uri":    best["uri"],
                "wikidata_label":  best["label"],
                "description":     best["description"][:100],
                "confidence":      best["score"],
                "status":          "MATCHED",
            })
            matched += 1

        else:
            # No match → local ontological definition 
            # The entity exists in our domain but not in Wikidata
            g_new.add((local_uri, RDF.type,     OWL.NamedIndividual))
            g_new.add((local_uri, RDFS.label,   Literal(label, lang="en")))
            g_new.add((local_uri, RDFS.comment, Literal(f"Entity from AI research domain, not found in Wikidata", lang="en")))

            # Add zero-confidence property
            best_score = best["score"] if best else 0.0
            best_candidate = best["label"] if best else "none"

            mapping_rows.append({
                "private_entity":  str(local_uri),
                "entity_label":    label,
                "ner_type":        ner_type,
                "wikidata_uri":    "",
                "wikidata_label":  best_candidate,
                "description":     "",
                "confidence":      best_score,
                "status":          "NOT_FOUND",
            })
            unmatched += 1

    save_cache(cache)

    # Export
    align_path = OUTPUT_DIR / "alignment_entities.ttl"
    new_path   = OUTPUT_DIR / "new_entities.ttl"
    csv_path   = OUTPUT_DIR / "mapping_table.csv"

    g_align.serialize(destination=str(align_path), format="turtle")
    g_new.serialize(destination=str(new_path), format="turtle")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["private_entity","entity_label","ner_type","wikidata_uri",
                      "wikidata_label","description","confidence","status"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(mapping_rows)

    print(f"\n{'='*50}")
    print(f"  Matched entities   : {matched:>6,}  (score >= {CONFIDENCE_THRESHOLD})")
    print(f"  Unmatched entities : {unmatched:>5,}")
    print(f"  Total              : {matched+unmatched:>6,}")
    print(f"{'='*50}")
    print(f"\n  → {align_path}")
    print(f"  → {new_path}")
    print(f"  → {csv_path}")



In [30]:
#Main
if __name__ == "__main__":
    kb_path = OUTPUT_DIR / "private_kb.nt"
    if not kb_path.exists():
        print("[ERROR] File private_kb.nt not found. Run step1_build_rdf.py first")
        exit(1)

    run_entity_linking(kb_path)


[STEP 2] Loading KB: output\private_kb.nt
  ✓ 377 entities to link
  [INFO] Clearing 67 failed cache entries (retrying)
  ... 20/377 entities processed
  ... 40/377 entities processed
  ... 60/377 entities processed
  ... 80/377 entities processed
  ... 100/377 entities processed
  ... 120/377 entities processed
  ... 140/377 entities processed
  ... 160/377 entities processed
  ... 180/377 entities processed
  ... 200/377 entities processed
  ... 220/377 entities processed
  ... 240/377 entities processed
  ... 260/377 entities processed
  ... 280/377 entities processed
  ... 300/377 entities processed
  ... 320/377 entities processed
  ... 340/377 entities processed
  ... 360/377 entities processed

  Matched entities   :    281  (score >= 0.75)
  Unmatched entities :    96
  Total              :    377

  → output\alignment_entities.ttl
  → output\new_entities.ttl
  → output\mapping_table.csv


### Step 2 : Results

| Category | Count | Rate |
|----------|-------|------|
| Matched (score ≥ 0.75) | 281 | 74.5% |
| Not found | 96 | 25.5% |
| **Total** | **377** | — |

The 74.5% match rate is consistent with expectations for this type of corpus:

- **High match rate for PERSON** (~76%): well-known AI researchers  are well represented in Wikidata.
- **Lower match rate for ORG** (~35%): specialized research groups
  are often absent from Wikidata.
- **Unmatched entities** include partial names , acronyms (`"CNNs"`),
  and domain-specific tools not in Wikidata.

The 96 unmatched entities are declared as `owl:NamedIndividual` in `output/new_entities.ttl`,
with a local ontological definition.

 All mappings are recorded in `output/mapping_table.csv` for report documentation.

### Step 3 : Predicate Alignment Using SPARQL

#### Objective
Map local predicates to standard Wikidata properties using either:
- `owl:equivalentProperty` — when the semantics are identical
- `rdfs:subPropertyOf` — when the local predicate is more specific or more general

In [31]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
DELAY = 1.5  
MAX_CANDIDATES  = 10

In [32]:
#Namespaces
ONTO  = Namespace("http://ai-kb.local/ontology/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
WD = Namespace("http://www.wikidata.org/entity/")
WB = Namespace("http://wikiba.se/ontology#")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [33]:
LOCAL_PREDICATES = [
    {
        "local_uri":     str(ONTO.relatedTo),
        "local_label":   "relatedTo",
        "keywords":      ["related", "associated", "connected"],
        "align_type":    "subProperty",
        "wikidata_prop": "P1659",   # see also
        "wikidata_label":"see also",
        "validated":     True,
        "rationale":     "co-occurrence extraction → generic relation, sub-property of P1659",
    },
    {
        "local_uri":     str(ONTO.affiliatedWith),
        "local_label":   "affiliatedWith",
        "keywords":      ["affiliation", "member", "employed"],
        "align_type":    "equivalent",
        "wikidata_prop": "P1416",   # affiliation
        "wikidata_label":"affiliation",
        "validated":     True,
        "rationale":     "affiliatedWith ≡ wdt:P1416 (affiliation)",
    },
    {
        "local_uri":     str(ONTO.worksAt),
        "local_label":   "worksAt",
        "keywords":      ["employer", "work", "employed by"],
        "align_type":    "equivalent",
        "wikidata_prop": "P108",    # employer
        "wikidata_label":"employer",
        "validated":     True,
        "rationale":     "worksAt ≡ wdt:P108 (employer)",
    },
    {
        "local_uri":     str(ONTO.authorOf),
        "local_label":   "authorOf",
        "keywords":      ["author", "wrote", "publication"],
        "align_type":    "equivalent",
        "wikidata_prop": "P50",     # author
        "wikidata_label":"author",
        "validated":     True,
        "rationale":     "authorOf ≡ wdt:P50 (author) — inverse of the standard property",
    },
    {
        "local_uri":     str(ONTO.locatedIn),
        "local_label":   "locatedIn",
        "keywords":      ["located", "country", "headquarters"],
        "align_type":    "equivalent",
        "wikidata_prop": "P17",     # country
        "wikidata_label":"country",
        "validated":     True,
        "rationale":     "locatedIn ≡ wdt:P17 (country) for organizations",
    },
]

In [34]:
def find_wikidata_predicates(keywords: list[str]) -> list[dict]:
    """
    Searches for Wikidata properties whose label contains the given keywords.
    Returns a list of candidate properties.
    
    """
    # Use the first keyword for the main query
    keyword = keywords[0].lower()

    query = f"""
    SELECT ?prop ?propLabel ?propDescription WHERE {{
      ?prop a wikibase:Property .
      ?prop rdfs:label ?propLabel .
      OPTIONAL {{ ?prop schema:description ?propDescription .
                  FILTER(LANG(?propDescription) = "en") }}
      FILTER(CONTAINS(LCASE(?propLabel), "{keyword}"))
      FILTER(LANG(?propLabel) = "en")
    }}
    LIMIT {MAX_CANDIDATES}
    """

    headers = {
        "Accept":     "application/sparql-results+json",
        "User-Agent": "AI-KB-Builder/1.0 (student project; educational use)",
    }

    try:
        resp = requests.get(
            WIKIDATA_SPARQL,
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"  [WARNING] SPARQL error: {e}")
        return []

    candidates = []
    for binding in data.get("results", {}).get("bindings", []):
        prop_uri = binding.get("prop", {}).get("value", "")
        # Extract the Pxxx identifier from the URI
        prop_id  = prop_uri.split("/")[-1] if prop_uri else ""
        label    = binding.get("propLabel", {}).get("value", "")
        desc     = binding.get("propDescription", {}).get("value", "")

        candidates.append({
            "wikidata_id":  prop_id,
            "wikidata_uri": f"http://www.wikidata.org/prop/direct/{prop_id}",
            "label":        label,
            "description":  desc,
        })

    time.sleep(DELAY)
    return candidates

In [35]:
def build_alignment_graph(predicates: list[dict]) -> tuple[Graph, list[dict]]:
    g = Graph()
    g.bind("onto", ONTO)
    g.bind("wdt",  WDT)
    g.bind("owl",  OWL)
    g.bind("rdfs", RDFS)

    # Declaration of local properties
    for pred in predicates:
        local_uri = URIRef(pred["local_uri"])
        g.add((local_uri, RDF.type,   OWL.ObjectProperty))
        g.add((local_uri, RDFS.label, Literal(pred["local_label"], lang="en")))

    candidates_report = []

    for pred in predicates:
        local_uri = URIRef(pred["local_uri"])

        print(f"\n  [SPARQL] Searching alignment for: {pred['local_label']}")

        if pred.get("validated") and pred.get("wikidata_prop"):
            # Already validated alignment: directly create the triple
            wd_uri = WDT[pred["wikidata_prop"]]

            if pred["align_type"] == "equivalent":
                g.add((local_uri, OWL.equivalentProperty, wd_uri))
                print(f"    ✓ owl:equivalentProperty → wdt:{pred['wikidata_prop']} ({pred['wikidata_label']})")
            elif pred["align_type"] == "subProperty":
                g.add((local_uri, RDFS.subPropertyOf, wd_uri))
                print(f"    ✓ rdfs:subPropertyOf → wdt:{pred['wikidata_prop']} ({pred['wikidata_label']})")

            # Justification comment
            g.add((local_uri, RDFS.comment, Literal(pred.get("rationale",""), lang="en")))

            candidates_report.append({
                "local_predicate":  pred["local_label"],
                "local_uri":        pred["local_uri"],
                "alignment_type":   pred["align_type"],
                "wikidata_id":      pred["wikidata_prop"],
                "wikidata_label":   pred["wikidata_label"],
                "status":           "VALIDATED",
                "rationale":        pred.get("rationale",""),
            })
        else:
            # SPARQL query to find candidate properties
            candidates = find_wikidata_predicates(pred["keywords"])

            if candidates:
                print(f"    Candidates found:")
                for c in candidates[:5]:
                    print(f"      wdt:{c['wikidata_id']:6s} | {c['label']:30s} | {c['description'][:60]}")

            candidates_report.append({
                "local_predicate":  pred["local_label"],
                "local_uri":        pred["local_uri"],
                "alignment_type":   "TO_VALIDATE",
                "wikidata_id":      candidates[0]["wikidata_id"] if candidates else "",
                "wikidata_label":   candidates[0]["label"] if candidates else "",
                "status":           "NEEDS_VALIDATION",
                "rationale":        "See SPARQL candidates above",
            })

    return g, candidates_report

In [36]:
def run_predicate_alignment():
    print("\n[STEP 3] Predicate alignment...")

    g, report = build_alignment_graph(LOCAL_PREDICATES)

    align_path = OUTPUT_DIR / "alignment_predicates.ttl"
    csv_path   = OUTPUT_DIR / "predicate_candidates.csv"

    g.serialize(destination=str(align_path), format="turtle")

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["local_predicate","local_uri","alignment_type",
                      "wikidata_id","wikidata_label","status","rationale"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(report)

    validated = sum(1 for r in report if r["status"] == "VALIDATED")
    pending   = sum(1 for r in report if r["status"] == "NEEDS_VALIDATION")

    print(f"\n{'='*50}")
    print(f"  Aligned predicates : {validated}")
    print(f"  To validate        : {pending}")
    print(f"{'='*50}")
    print(f"\n  → {align_path}")
    print(f"  → {csv_path}")

In [37]:
if __name__ == "__main__":
    run_predicate_alignment()


[STEP 3] Predicate alignment...

  [SPARQL] Searching alignment for: relatedTo
    ✓ rdfs:subPropertyOf → wdt:P1659 (see also)

  [SPARQL] Searching alignment for: affiliatedWith
    ✓ owl:equivalentProperty → wdt:P1416 (affiliation)

  [SPARQL] Searching alignment for: worksAt
    ✓ owl:equivalentProperty → wdt:P108 (employer)

  [SPARQL] Searching alignment for: authorOf
    ✓ owl:equivalentProperty → wdt:P50 (author)

  [SPARQL] Searching alignment for: locatedIn
    ✓ owl:equivalentProperty → wdt:P17 (country)

  Aligned predicates : 5
  To validate        : 0

  → output\alignment_predicates.ttl
  → output\predicate_candidates.csv


### Step 4 : KB Expansion via SPARQL



#### Objective
Scale the KB from ~2,500 triples to **50,000–200,000 triples** by querying Wikidata
for entities related to our anchored entities.

#### Anchored Expansion Strategy
Only entities with a confident `owl:sameAs` link (281 entities) are used as expansion anchors.
This ensures the expanded KB remains **thematically coherent** with the AI Research domain.

#### Expansion Phases

| Phase | Strategy | SPARQL Pattern |
|-------|----------|----------------|
| 1 | 1-hop from anchored entities | `wd:Q{id} ?p ?o` |
| 2 | AI researchers by field of work | `?person wdt:P101 wd:Q11660` |
| 3 | ML/AI organizations | `?org wdt:P31 wd:Q31855` (research institute) |
| 4 | 2-hop colleagues (same org) | `?anchor wdt:P108 ?org . ?colleague wdt:P108 ?org` |
| 5 | Expansion by key predicates | P166 (awards), P69 (education), P108 (employer) |

#### Predicate Whitelist
To avoid polluting the KB with Wikidata metadata, only **25 informative predicates** are allowed:
`P108, P50, P17, P131, P1416, P69, P166, P31, P106, P19, P27, P571, P749, P355, P361...`

#### Size Control
Each query uses a strict `LIMIT`. Expansion stops automatically when `TARGET_MAX_TRIPLES` is reached.

In [50]:
TARGET_MIN_TRIPLES  = 50_000    
TARGET_MAX_TRIPLES  = 200_000  
DELAY = 1.0      
USE_CACHE = True
BATCH_SAVE_EVERY = 50    

In [51]:
# Allowed predicates for expansion 
# Excludes overly verbose predicates (statements, labels, Wikidata descriptions)
ALLOWED_PREDICATES = {
    "P108",  "P50",   "P17",   "P131",  "P1416",
    "P495",  "P571",  "P31",   "P106",  "P19",
    "P69",   "P166",  "P1344", "P2860", "P527",
    "P20",   "P27",   "P159",  "P749",  "P355",
    "P361",  "P460",  "P21",   "P1308", "P800",
}

In [52]:
#Namespaces
LOCAL  = Namespace("http://ai-kb.local/entity/")
ONTO   = Namespace("http://ai-kb.local/ontology/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

OUTPUT_DIR  = Path("output")
CACHE_FILE  = OUTPUT_DIR / "expansion_cache.json"
OUTPUT_NT   = OUTPUT_DIR / "expanded_kb.nt"
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "Accept":     "application/sparql-results+json",
    "User-Agent": "AI-KB-Builder/1.0 (student project; educational use)",
}


In [53]:
#Cache
def load_cache():
    if USE_CACHE and CACHE_FILE.exists():
        with open(CACHE_FILE) as f:
            return json.load(f)
    return {}


def save_cache(cache: dict):
    if USE_CACHE:
        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f, ensure_ascii=False)

In [54]:
#SPARQL helpers
def sparql_query(query: str, cache: dict, cache_key: str) -> list[dict]:
    """
    Executes a SPARQL query on Wikidata.
    Returns a list of bindings (dicts).
    """
    if cache_key in cache:
        return cache[cache_key]

    try:
        resp = requests.get(
            WIKIDATA_SPARQL,
            params={"query": query, "format": "json"},
            headers=HEADERS,
            timeout=60,
        )
        if resp.status_code == 429:
            print("  [WARNING] Wikidata rate limit, waiting 30s...")
            time.sleep(30)
            resp = requests.get(
                WIKIDATA_SPARQL,
                params={"query": query, "format": "json"},
                headers=HEADERS,
                timeout=60
            )
        resp.raise_for_status()
        bindings = resp.json().get("results", {}).get("bindings", [])
    except Exception as e:
        print(f"  [WARNING] SPARQL error ({cache_key[:40]}): {e}")
        bindings = []

    cache[cache_key] = bindings
    time.sleep(DELAY)
    return bindings

In [55]:
#Entities extraction
def get_anchored_entities(align_path: Path) -> list[tuple[URIRef, str]]:
    """
    Reads the alignment file to extract pairs (local_URI, Wikidata_QID).
    
    """
    g = Graph()
    g.parse(str(align_path), format="turtle")

    anchored = []
    for subj, _, obj in g.triples((None, OWL.sameAs, None)):
        if "wikidata.org/entity/Q" in str(obj):
            qid = str(obj).split("/")[-1]
            anchored.append((subj, qid))

    print(f"  ✓ {len(anchored)} anchored entities found")
    return anchored

In [56]:
def expand_1hop(qid: str, cache: dict) -> list[tuple]:
    """
    1-hop expansion: all direct triples 
    Filtered using ALLOWED_PREDICATES to avoid noise.
    """
    allowed_str = " ".join(f"wdt:{p}" for p in ALLOWED_PREDICATES)
    query = f"""
    SELECT ?p ?o WHERE {{
      VALUES ?p {{ {allowed_str} }}
      wd:{qid} ?p ?o .
      FILTER(!isLiteral(?o) || LANG(?o) = "" || LANG(?o) = "en")
    }}
    LIMIT 200
    """
    bindings = sparql_query(query, cache, f"1hop_{qid}")
    triples = []
    for b in bindings:
        p_uri = b.get("p", {}).get("value", "")
        o_val = b.get("o", {}).get("value", "")
        o_type = b.get("o", {}).get("type", "")
        if p_uri and o_val:
            triples.append((qid, p_uri, o_val, o_type))
    return triples


def expand_by_predicate(wdt_prop: str, limit: int, cache: dict) -> list[tuple]:
    """
    Predicate-based expansion: all subjects/objects linked by this predicate.

    """
    query = f"""
    SELECT ?s ?o WHERE {{
      ?s wdt:{wdt_prop} ?o .
      ?s wdt:P31 wd:Q5 .   # instance of human (avoids bots/noisy entities)
    }}
    LIMIT {limit}
    """
    bindings = sparql_query(query, cache, f"bypred_{wdt_prop}_{limit}")
    triples = []
    for b in bindings:
        s_val = b.get("s", {}).get("value", "")
        o_val = b.get("o", {}).get("value", "")
        if s_val and o_val:
            triples.append((s_val, f"http://www.wikidata.org/prop/direct/{wdt_prop}", o_val, "uri"))
    return triples


def expand_2hop_organizations(anchored_qids: list[str], cache: dict, limit=5000) -> list[tuple]:
    """
    2-hop expansion: finds people working in the same organizations
    as the anchored entities.

    """
    if not anchored_qids:
        return []

    qids_str = " ".join(f"wd:{q}" for q in anchored_qids[:50])  # max 50 anchors
    query = f"""
    SELECT DISTINCT ?person ?org ?p WHERE {{
      VALUES ?anchor {{ {qids_str} }}
      VALUES ?p {{ wdt:P108 wdt:P1416 }}
      ?anchor ?p ?org .
      ?person ?p ?org .
      FILTER(?person != ?anchor)
    }}
    LIMIT {limit}
    """
    bindings = sparql_query(query, cache, f"2hop_orgs_{len(anchored_qids)}")
    triples = []
    for b in bindings:
        s_val = b.get("person", {}).get("value", "")
        o_val = b.get("org", {}).get("value", "")
        p_val = b.get("p", {}).get("value", "")
        if s_val and o_val and p_val:
            triples.append((s_val, p_val, o_val, "uri"))
    return triples


def expand_ai_researchers(cache: dict, limit=10000) -> list[tuple]:
    """
    Thematic expansion: finds AI/ML researchers in Wikidata.
    Uses P106 (occupation) = Q82594 (computer scientist) or Q1622272 (university teacher)
    with P101 (field of work) = Q11660 (artificial intelligence).
    """
    query = f"""
    SELECT DISTINCT ?person ?p ?o WHERE {{
      ?person wdt:P101 wd:Q11660 .    # field of work = artificial intelligence
      VALUES ?p {{ wdt:P108 wdt:P1416 wdt:P69 wdt:P166 wdt:P19 wdt:P27 }}
      ?person ?p ?o .
    }}
    LIMIT {limit}
    """
    bindings = sparql_query(query, cache, f"ai_researchers_{limit}")
    triples = []
    for b in bindings:
        s_val = b.get("person", {}).get("value", "")
        p_val = b.get("p", {}).get("value", "")
        o_val = b.get("o", {}).get("value", "")
        if s_val and p_val and o_val:
            triples.append((s_val, p_val, o_val, "uri"))
    return triples


def expand_ml_organizations(cache: dict, limit=5000) -> list[tuple]:
    """
    Organizations related to AI/ML: universities, labs, tech companies.
    """
    query = f"""
    SELECT DISTINCT ?org ?p ?o WHERE {{
      ?org wdt:P31 ?type .
      VALUES ?type {{ wd:Q31855 wd:Q7210356 wd:Q245065 wd:Q1935049 }}
      # types: research institute, multinational corporation, etc.
      VALUES ?p {{ wdt:P17 wdt:P131 wdt:P749 wdt:P355 wdt:P571 }}
      ?org ?p ?o .
    }}
    LIMIT {limit}
    """
    bindings = sparql_query(query, cache, f"ml_orgs_{limit}")
    triples = []
    for b in bindings:
        s_val = b.get("org",{}).get("value","")
        p_val = b.get("p",{}).get("value","")
        o_val = b.get("o",{}).get("value","")
        if s_val and p_val and o_val:
            triples.append((s_val, p_val, o_val, "uri"))
    return triples

In [57]:
def add_raw_triples_to_graph(g: Graph, raw_triples: list[tuple]):
    """
    Adds raw triples into the RDF graph.
    Format: (subject_wd_uri_or_qid, predicate_uri, object_val, object_type)
    """
    added = 0
    for item in raw_triples:
        s_raw, p_raw, o_raw, o_type = item

        # Subject normalization
        if s_raw.startswith("http://"):
            s_uri = URIRef(s_raw)
        elif s_raw.startswith("Q"):
            s_uri = WD[s_raw]
        else:
            continue

        # Predicate normalization
        p_uri = URIRef(p_raw) if p_raw.startswith("http://") else None
        if p_uri is None:
            continue

        # Object normalization
        if o_type == "uri" or o_raw.startswith("http://"):
            o_node = URIRef(o_raw)
        else:
            o_node = Literal(o_raw)

        g.add((s_uri, p_uri, o_node))
        added += 1
    return added

In [58]:
def run_expansion():
    cache = load_cache()

    # Load the private KB
    print("\n[STEP 4] Loading the private KB...")
    g = Graph()
    g.bind("wd",  WD)
    g.bind("wdt", WDT)
    g.bind("local", LOCAL)

    kb_path = OUTPUT_DIR / "private_kb.nt"
    align_path = OUTPUT_DIR / "alignment_entities.ttl"

    if kb_path.exists():
        g.parse(str(kb_path), format="nt")
        print(f"  ✓ Private KB loaded: {len(g)} triples")

    if align_path.exists():
        g.parse(str(align_path), format="turtle")
        print(f"  ✓ Alignments loaded: {len(g)} total triples")

    # Retrieve anchored entities
    anchored = get_anchored_entities(align_path) if align_path.exists() else []
    anchored_qids = [qid for _, qid in anchored]

    print(f"\n[STEP 4] Starting expansion (target: {TARGET_MIN_TRIPLES:,}–{TARGET_MAX_TRIPLES:,} triples)")
    print(f"  Current triples: {len(g):,}")

    total_added = 0

    # Phase 1: 1-hop expansion on anchored entities 
    print(f"\n  Phase 1: 1-hop expansion ({len(anchored_qids)} anchored entities)...")
    for i, qid in enumerate(anchored_qids):
        if len(g) >= TARGET_MAX_TRIPLES:
            print("  ⚠️  MAX limit reached, stopping.")
            break

        raw = expand_1hop(qid, cache)
        n   = add_raw_triples_to_graph(g, raw)
        total_added += n

        if (i+1) % BATCH_SAVE_EVERY == 0:
            print(f"    ... {i+1}/{len(anchored_qids)} | {len(g):,} total triples")
            save_cache(cache)

    save_cache(cache)
    print(f"  ✓ After Phase 1: {len(g):,} triples")

    # Phase 2: Thematic expansion — AI Researchers 
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        limit  = min(needed * 2, 30_000)
        print(f"\n  Phase 2: AI researchers (limit={limit:,})...")
        raw = expand_ai_researchers(cache, limit=limit)
        n   = add_raw_triples_to_graph(g, raw)
        total_added += n
        save_cache(cache)
        print(f"  ✓ +{n:,} triples | Total: {len(g):,}")

    # Phase 3: ML organizations 
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        limit  = min(needed * 2, 20_000)
        print(f"\n  Phase 3: ML organizations (limit={limit:,})...")
        raw = expand_ml_organizations(cache, limit=limit)
        n   = add_raw_triples_to_graph(g, raw)
        total_added += n
        save_cache(cache)
        print(f"  ✓ +{n:,} triples | Total: {len(g):,}")

    # Phase 4: 2-hop via shared organizations 
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        limit  = min(needed, 10_000)
        print(f"\n  Phase 4: 2-hop expansion (colleagues, limit={limit:,})...")
        raw = expand_2hop_organizations(anchored_qids, cache, limit=limit)
        n   = add_raw_triples_to_graph(g, raw)
        total_added += n
        save_cache(cache)
        print(f"  ✓ +{n:,} triples | Total: {len(g):,}")

    # Phase 5: Expansion by important predicates 
    PRIORITY_PREDICATES = [
        ("P166", 10000),   # award received — links persons ↔ awards
        ("P69",  10000),   # educated at — links persons ↔ universities
        ("P108", 15000),   # employer — links persons ↔ organizations
    ]
    for wdt_prop, limit in PRIORITY_PREDICATES:
        if len(g) >= TARGET_MAX_TRIPLES:
            print("  MAX limit reached, stopping Phase 5.")
            break
        print(f"\n  Phase 5: Expansion wdt:{wdt_prop} (limit={limit:,})...")
        raw = expand_by_predicate(wdt_prop, limit, cache)
        n   = add_raw_triples_to_graph(g, raw)
        total_added += n
        save_cache(cache)
        print(f"  + {n:,} triples | Total: {len(g):,}")

    # Phase 6: ML/DL publications
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        lim = min(needed, 20_000)
        print(f"\n  Phase 6: ML publications (P2860 cites work, limit={lim:,})...")
        query = f"""
        SELECT DISTINCT ?s ?o WHERE {{
          ?s wdt:P101 wd:Q11660 .
          ?s wdt:P2860 ?o .
        }}
        LIMIT {lim}
        """
        bindings = sparql_query(query, cache, f"publications_{lim}")
        raw = [(b["s"]["value"], "http://www.wikidata.org/prop/direct/P2860", b["o"]["value"], "uri")
               for b in bindings if b.get("s") and b.get("o")]
        n = add_raw_triples_to_graph(g, raw)
        save_cache(cache)
        print(f"  + {n:,} triples | Total: {len(g):,}")

    # Phase 7: Computer scientists (P106 = Q82594)
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        lim = min(needed * 2, 25_000)
        print(f"\n  Phase 7: Computer scientists (P106=Q82594, limit={lim:,})...")
        query = f"""
        SELECT DISTINCT ?person ?p ?o WHERE {{
          ?person wdt:P106 wd:Q82594 .
          VALUES ?p {{ wdt:P108 wdt:P69 wdt:P19 wdt:P27 wdt:P166 wdt:P1416 }}
          ?person ?p ?o .
        }}
        LIMIT {lim}
        """
        bindings = sparql_query(query, cache, f"cs_researchers_{lim}")
        raw = [(b["person"]["value"], b["p"]["value"], b["o"]["value"], "uri")
               for b in bindings if b.get("person") and b.get("p") and b.get("o")]
        n = add_raw_triples_to_graph(g, raw)
        save_cache(cache)
        print(f"  + {n:,} triples | Total: {len(g):,}")

    # Phase 8: NLP/ML researchers (P101 = Q8142 NLP or Q2539 ML) 
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        lim = min(needed * 2, 20_000)
        print(f"\n  Phase 8: NLP + ML researchers (limit={lim:,})...")
        query = f"""
        SELECT DISTINCT ?person ?p ?o WHERE {{
          ?person wdt:P101 ?field .
          VALUES ?field {{ wd:Q2539 wd:Q8142 wd:Q30642 wd:Q11660 }}
          VALUES ?p {{ wdt:P108 wdt:P69 wdt:P19 wdt:P27 wdt:P1416 wdt:P166 }}
          ?person ?p ?o .
        }}
        LIMIT {lim}
        """
        bindings = sparql_query(query, cache, f"nlp_ml_{lim}")
        raw = [(b["person"]["value"], b["p"]["value"], b["o"]["value"], "uri")
               for b in bindings if b.get("person") and b.get("p") and b.get("o")]
        n = add_raw_triples_to_graph(g, raw)
        save_cache(cache)
        print(f"  + {n:,} triples | Total: {len(g):,}")

    # Phase 9: Top tech companies employees
    if len(g) < TARGET_MIN_TRIPLES:
        needed = TARGET_MIN_TRIPLES - len(g)
        lim = min(needed, 15_000)
        print(f"\n  Phase 9: Employees of tech/AI organizations (limit={lim:,})...")
        # OpenAI=Q21708100, Google=Q95, DeepMind=Q15733006, Meta=Q380, Microsoft=Q2283
        query = f"""
        SELECT DISTINCT ?person ?p ?o WHERE {{
          VALUES ?org {{ wd:Q21708100 wd:Q95 wd:Q15733006 wd:Q380 wd:Q2283 wd:Q484696 wd:Q1155395 }}
          ?person wdt:P108 ?org .
          VALUES ?p {{ wdt:P108 wdt:P69 wdt:P19 wdt:P27 wdt:P166 wdt:P1416 wdt:P106 }}
          ?person ?p ?o .
        }}
        LIMIT {lim}
        """
        bindings = sparql_query(query, cache, f"tech_employees_{lim}")
        raw = [(b["person"]["value"], b["p"]["value"], b["o"]["value"], "uri")
               for b in bindings if b.get("person") and b.get("p") and b.get("o")]
        n = add_raw_triples_to_graph(g, raw)
        save_cache(cache)
        print(f"  + {n:,} triples | Total: {len(g):,}")

    # Final export 
    print(f"\n[STEP 4] Exporting the expanded KB...")
    g.serialize(destination=str(OUTPUT_NT), format="nt")

    # Statistics
    all_subjects = set(g.subjects())
    all_objects  = set(o for o in g.objects() if isinstance(o, URIRef))
    all_entities = all_subjects | all_objects
    all_preds    = set(g.predicates())

    stats = f"""
{'='*55}
  EXPANDED KB STATISTICS
{'='*55}
  Total triples      : {len(g):>10,}
  Entities (nodes)   : {len(all_entities):>10,}
  Unique predicates  : {len(all_preds):>10,}
{'='*55}
  Min target : {TARGET_MIN_TRIPLES:>10,}   -> {'OK' if len(g) >= TARGET_MIN_TRIPLES else 'INSUFFICIENT - rerun with more triples'}
  Max target : {TARGET_MAX_TRIPLES:>10,}   -> {'OK' if len(g) <= TARGET_MAX_TRIPLES else 'EXCEEDED'}
{'='*55}
"""
    print(stats)

    stats_path = OUTPUT_DIR / "expansion_stats.txt"
    with open(stats_path, "w", encoding="utf-8") as f:
        f.write(stats.encode("ascii","replace").decode("ascii"))

    print(f"  → {OUTPUT_NT}")
    print(f"  → {stats_path}")

In [59]:
#Main
if __name__ == "__main__":
    run_expansion()


[STEP 4] Loading the private KB...
  ✓ Private KB loaded: 2246 triples
  ✓ Alignments loaded: 2539 total triples
  ✓ 281 anchored entities found

[STEP 4] Starting expansion (target: 50,000–200,000 triples)
  Current triples: 2,539

  Phase 1: 1-hop expansion (281 anchored entities)...
    ... 50/281 | 2,723 total triples
  [WARNING] SPARQL error (1hop_Q37007847): HTTPSConnectionPool(host='query.wikidata.org', port=443): Read timed out. (read timeout=60)
    ... 100/281 | 2,933 total triples
    ... 150/281 | 3,306 total triples
    ... 200/281 | 3,660 total triples
    ... 250/281 | 3,823 total triples
  ✓ After Phase 1: 3,951 triples

  Phase 2: AI researchers (limit=30,000)...
  ✓ +5,769 triples | Total: 9,692

  Phase 3: ML organizations (limit=20,000)...
  ✓ +20,000 triples | Total: 29,692

  Phase 4: 2-hop expansion (colleagues, limit=10,000)...
  ✓ +511 triples | Total: 30,188

  Phase 5: Expansion wdt:P166 (limit=10,000)...
  + 10,000 triples | Total: 40,185

  Phase 5: Expans

### Cleaning Before Embedding

In [73]:
import random
from collections import Counter, defaultdict
from pathlib import Path

from rdflib import Graph, URIRef, Literal, RDF, RDFS, OWL
import networkx as nx

In [74]:
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42

In [75]:
# Predicates to EXCLUDE from KGE 
EXCLUDED_PREDICATES = {
    # Structural RDF/RDFS/OWL predicates
    str(RDF.type),
    str(RDFS.label),
    str(RDFS.comment),
    str(OWL.sameAs),
    str(OWL.equivalentProperty),
    str(RDFS.subPropertyOf),
    # Wikidata metadata predicates
    "http://www.wikidata.org/prop/direct/P31",    # instance of (keep for KGE if desired)
    "http://schema.org/description",
    "http://www.w3.org/2004/02/skos/core#altLabel",
    "http://www.w3.org/2004/02/skos/core#prefLabel",
    # Local provenance predicates
    "http://ai-kb.local/ontology/extractedFrom",
    "http://ai-kb.local/ontology/entityLabel",
}

# Maximum literal length (beyond this, it is a description and not useful for KGE)
MAX_LITERAL_LENGTH = 200

OUTPUT_DIR = Path("output")
KGE_DIR    = OUTPUT_DIR / "kge"
KGE_DIR.mkdir(parents=True, exist_ok=True)

In [76]:
def load_and_filter_graph(nt_path: Path) -> list[tuple[str, str, str]]:
    """
    Loads the NT file, applies filters, and returns a list of clean triples.
    """
    print(f"\n[STEP 5] Loading {nt_path}...")
    g = Graph()
    g.parse(str(nt_path), format="nt")
    print(f"  ✓ {len(g):,} triples loaded")

    clean_triples = []
    skipped = Counter()

    for s, p, o in g:
        # Filter 1: excluded predicates
        if str(p) in EXCLUDED_PREDICATES:
            skipped["excluded_pred"] += 1
            continue

        # Filter 2: literals that are too long
        if isinstance(o, Literal) and len(str(o)) > MAX_LITERAL_LENGTH:
            skipped["long_literal"] += 1
            continue

        # Filter 3: empty or malformed URIs
        if isinstance(s, URIRef) and len(str(s)) < 10:
            skipped["bad_uri"] += 1
            continue

        # Filter 4: non-English literals (reduce language noise)
        if isinstance(o, Literal) and hasattr(o, 'language') and o.language not in (None, "en", ""):
            skipped["non_en_literal"] += 1
            continue

        # Encoding: URIRef → string, Literal → prefixed string
        s_str = str(s)
        p_str = str(p)
        o_str = str(o) if isinstance(o, URIRef) else f"__literal__{str(o)}"

        clean_triples.append((s_str, p_str, o_str))

    # Deduplication
    clean_triples = list(set(clean_triples))

    print(f"  Filters applied:")
    for reason, count in skipped.most_common():
        print(f"    {reason:25s} : {count:>8,}")
    print(f"  ✓ {len(clean_triples):,} triples after cleaning")

    return clean_triples

In [77]:
def check_connectivity(triples: list[tuple]) -> dict:
    """
    Builds an undirected NetworkX graph and analyzes connected components.
    Returns statistics.
    """
    print("\n[STEP 5] Connectivity analysis...")

    # We work only with URI entities (not literals)
    uri_triples = [(s, o) for s, p, o in triples if not o.startswith("__literal__")]

    G = nx.Graph()
    for s, o in uri_triples:
        G.add_edge(s, o)

    components = list(nx.connected_components(G))
    comp_sizes = sorted([len(c) for c in components], reverse=True)

    stats = {
        "n_nodes":          G.number_of_nodes(),
        "n_edges":          G.number_of_edges(),
        "n_components":     len(components),
        "largest_component":comp_sizes[0] if comp_sizes else 0,
        "isolated_nodes":   sum(1 for s in comp_sizes if s == 1),
    }

    pct_in_main = stats["largest_component"] / max(stats["n_nodes"], 1) * 100

    print(f"  Nodes            : {stats['n_nodes']:>10,}")
    print(f"  Edges            : {stats['n_edges']:>10,}")
    print(f"  Components       : {stats['n_components']:>10,}")
    print(f"  Largest comp.    : {stats['largest_component']:>10,}  ({pct_in_main:.1f}% of the graph)")
    print(f"  Isolated nodes   : {stats['isolated_nodes']:>10,}")

    if pct_in_main < 50:
        print("  ⚠️  WARNING: graph is highly fragmented. Increase 2-hop expansion.")
    else:
        print("  ✓ Connectivity is acceptable.")

    return stats

In [78]:
def filter_low_degree_entities(triples: list[tuple], min_degree: int = 2) -> list[tuple]:
    """
    Improve embedding quality (rare entities = noise)
    """
    degree = Counter()
    for s, p, o in triples:
        degree[s] += 1
        degree[o] += 1

    kept = [(s, p, o) for s, p, o in triples
            if degree[s] >= min_degree and degree[o] >= min_degree]

    removed = len(triples) - len(kept)
    print(f"\n[STEP 5] Filtrage degré < {min_degree} : {removed:,} triplets supprimés ({len(kept):,} conservés)")
    return kept

In [79]:
def build_dictionaries(triples: list[tuple]) -> tuple[dict, dict]:
    """
   Maps each unique URI to an integer (required by KGE frameworks).
    """
    entities  = set()
    relations = set()

    for s, p, o in triples:
        entities.add(s)
        entities.add(o)
        relations.add(p)

    entity2id   = {e: i for i, e in enumerate(sorted(entities))}
    relation2id = {r: i for i, r in enumerate(sorted(relations))}

    return entity2id, relation2id

In [80]:
def split_triples(triples: list[tuple]) -> tuple[list, list, list]:
    """
    Stratified split by relation to maintain a balanced distribution.
    """
    # Group by relation
    by_relation = defaultdict(list)
    for t in triples:
        by_relation[t[1]].append(t)

    train, valid, test = [], [], []
    rng = random.Random(RANDOM_SEED)

    for rel, rel_triples in by_relation.items():
        rng.shuffle(rel_triples)
        n = len(rel_triples)
        n_train = max(1, int(n * TRAIN_RATIO))
        n_valid = max(1, int(n * VALID_RATIO))

        train.extend(rel_triples[:n_train])
        valid.extend(rel_triples[n_train:n_train + n_valid])
        test.extend(rel_triples[n_train + n_valid:])

    return train, valid, test

In [81]:
def export_kge_files(
    triples_train: list,
    triples_valid: list,
    triples_test:  list,
    entity2id:     dict,
    relation2id:   dict,
    conn_stats:    dict,
):
    def write_triples(path: Path, triples: list):
        with open(path, "w", encoding="utf-8") as f:
            for s, p, o in triples:
                eid_s = entity2id.get(s, -1)
                rid_p = relation2id.get(p, -1)
                eid_o = entity2id.get(o, -1)
                if eid_s >= 0 and rid_p >= 0 and eid_o >= 0:
                    f.write(f"{eid_s}\t{rid_p}\t{eid_o}\n")

    write_triples(KGE_DIR / "train.txt", triples_train)
    write_triples(KGE_DIR / "valid.txt", triples_valid)
    write_triples(KGE_DIR / "test.txt",  triples_test)

    # Dictionaries
    with open(KGE_DIR / "entities.dict", "w", encoding="utf-8") as f:
        for uri, idx in sorted(entity2id.items(), key=lambda x: x[1]):
            f.write(f"{idx}\t{uri}\n")

    with open(KGE_DIR / "relations.dict", "w", encoding="utf-8") as f:
        for uri, idx in sorted(relation2id.items(), key=lambda x: x[1]):
            f.write(f"{idx}\t{uri}\n")

    # Final statistics report
    total = len(triples_train) + len(triples_valid) + len(triples_test)
    stats_text = f"""
{'='*55}
  FINAL REPORT — KB READY FOR KGE
{'='*55}

  SPLIT
  ─────
  Train      : {len(triples_train):>10,}  ({len(triples_train)/total*100:.1f}%)
  Validation : {len(triples_valid):>10,}  ({len(triples_valid)/total*100:.1f}%)
  Test       : {len(triples_test):>10,}  ({len(triples_test)/total*100:.1f}%)
  Total      : {total:>10,}

  DICTIONARIES
  ────────────
  Entities   : {len(entity2id):>10,}
  Relations  : {len(relation2id):>10,}

  CONNECTIVITY
  ────────────
  Nodes              : {conn_stats['n_nodes']:>10,}
  Components         : {conn_stats['n_components']:>10,}
  Largest component  : {conn_stats['largest_component']:>10,}
  Isolated nodes     : {conn_stats['isolated_nodes']:>10,}

  GENERATED FILES
  ───────────────
  {KGE_DIR}/train.txt
  {KGE_DIR}/valid.txt
  {KGE_DIR}/test.txt
  {KGE_DIR}/entities.dict
  {KGE_DIR}/relations.dict

  COMMANDS TO RUN KGE (example with PyKEEN)
  ────────────────────────────────────────
  pip install pykeen
  python -c "
  from pykeen.pipeline import pipeline
  result = pipeline(
      training='output/kge/train.txt',
      validation='output/kge/valid.txt',
      testing='output/kge/test.txt',
      model='TransE',
      training_kwargs=dict(num_epochs=100),
  )
  result.save_to_directory('kge_results/transe')
  "
{'='*55}
"""
    stats_path = KGE_DIR / "stats.txt"
    with open(stats_path, "w", encoding="utf-8") as f:
        f.write(stats_text)
    print(stats_text)

In [82]:
def run_cleaning():
    nt_path = OUTPUT_DIR / "expanded_kb.nt"
    if not nt_path.exists():
        # Fallback to the private KB if expansion was not performed
        nt_path = OUTPUT_DIR / "private_kb.nt"
        print(f"[INFO] expanded_kb.nt not found, using {nt_path}")

    triples = load_and_filter_graph(nt_path)
    conn_stats = check_connectivity(triples)

    # Optional filtering of low-degree entities
    # Comment this line if the graph is already small
    # Degree filter disabled (min_degree=1) to keep all triples
    # triples = filter_low_degree_entities(triples, min_degree=2)

    entity2id, relation2id = build_dictionaries(triples)
    triples_train, triples_valid, triples_test = split_triples(triples)

    export_kge_files(triples_train, triples_valid, triples_test,
                     entity2id, relation2id, conn_stats)

In [83]:
#Main
if __name__ == "__main__":
    run_cleaning()


[STEP 5] Loading output\expanded_kb.nt...
  ✓ 65,103 triples loaded
  Filters applied:
    excluded_pred             :    2,418
  ✓ 62,685 triples after cleaning

[STEP 5] Connectivity analysis...
  Nodes            :     50,196
  Edges            :     60,972
  Components       :        607
  Largest comp.    :     46,487  (92.6% of the graph)
  Isolated nodes   :          0
  ✓ Connectivity is acceptable.

  FINAL REPORT — KB READY FOR KGE

  SPLIT
  ─────
  Train      :     50,138  (80.0%)
  Validation :      6,260  (10.0%)
  Test       :      6,287  (10.0%)
  Total      :     62,685

  DICTIONARIES
  ────────────
  Entities   :     50,218
  Relations  :         24

  CONNECTIVITY
  ────────────
  Nodes              :     50,196
  Components         :        607
  Largest component  :     46,487
  Isolated nodes     :          0

  GENERATED FILES
  ───────────────
  output\kge/train.txt
  output\kge/valid.txt
  output\kge/test.txt
  output\kge/entities.dict
  output\kge/relations.

### Generate delivarables

In [84]:
from pathlib import Path
from collections import Counter
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

# Namespaces 
LOCAL = Namespace("http://ai-kb.local/entity/")
ONTO  = Namespace("http://ai-kb.local/ontology/")
WD    = Namespace("http://www.wikidata.org/entity/")
WDT   = Namespace("http://www.wikidata.org/prop/direct/")

OUTPUT_DIR = Path("output")
KGE_DIR    = OUTPUT_DIR / "kge"
OUTPUT_DIR.mkdir(exist_ok=True)


# KB statistics 
def compute_stats(nt_path: Path) -> dict:
    """Computes complete statistics for an NT file."""
    if not nt_path.exists():
        return {}

    g = Graph()
    g.parse(str(nt_path), format="nt")

    all_subjects   = set(g.subjects())
    all_objects    = set(o for o in g.objects() if isinstance(o, URIRef))
    all_entities   = all_subjects | all_objects
    all_predicates = set(g.predicates())

    # Predicate distribution
    pred_counts = Counter()
    for _, p, _ in g:
        pred_counts[str(p)] += 1

    # Top 10 predicates
    top_preds = pred_counts.most_common(10)

    # Entity types
    type_counts = Counter()
    for _, _, rdf_type in g.triples((None, RDF.type, None)):
        type_counts[str(rdf_type).split("/")[-1]] += 1

    return {
        "n_triples":   len(g),
        "n_entities":  len(all_entities),
        "n_relations": len(all_predicates),
        "n_subjects":  len(all_subjects),
        "n_objects":   len(all_objects),
        "top_predicates": top_preds,
        "type_distribution": type_counts.most_common(10),
    }


def compute_split_stats() -> dict:
    """Reads statistics for the KGE splits."""
    stats = {}
    for split in ["train", "valid", "test"]:
        f = KGE_DIR / f"{split}.txt"
        if f.exists():
            stats[split] = sum(1 for _ in open(f, encoding="utf-8"))
    return stats


# Final ontology 
def generate_ontology() -> Graph:
    """Generates a minimal and consistent OWL ontology for the KB."""
    g = Graph()
    g.bind("onto",  ONTO)
    g.bind("owl",   OWL)
    g.bind("rdfs",  RDFS)
    g.bind("xsd",   XSD)

    SCHEMA = Namespace("https://schema.org/")
    g.bind("schema", SCHEMA)

    # Ontology header
    onto_uri = URIRef("http://ai-kb.local/ontology/")
    g.add((onto_uri, RDF.type,      OWL.Ontology))
    g.add((onto_uri, RDFS.label,    Literal("AI Research Knowledge Base Ontology", lang="en")))
    g.add((onto_uri, RDFS.comment,  Literal("Ontology for the AI Research domain KB, built from web-mined data.", lang="en")))

    # Classes 
    classes = [
        (ONTO.Person,             "Person",             "A human researcher or scientist", SCHEMA.Person),
        (ONTO.Organization,       "Organization",       "A research institution, company, or university", SCHEMA.Organization),
        (ONTO.GeopoliticalEntity, "GeopoliticalEntity", "A country, city, or region", SCHEMA.Place),
        (ONTO.Date,               "Date",               "A temporal reference (year, period)", None),
        (ONTO.ResearchPaper,      "ResearchPaper",      "A published academic paper or preprint", SCHEMA.ScholarlyArticle),
        (ONTO.Award,              "Award",              "An academic or professional award", SCHEMA.Award),
    ]

    for cls_uri, label, comment, schema_super in classes:
        g.add((cls_uri, RDF.type,       OWL.Class))
        g.add((cls_uri, RDFS.label,     Literal(label, lang="en")))
        g.add((cls_uri, RDFS.comment,   Literal(comment, lang="en")))
        if schema_super:
            g.add((cls_uri, RDFS.subClassOf, schema_super))

    # Properties 
    obj_props = [
        # (URI, label, comment, domain, range)
        (ONTO.relatedTo,     "relatedTo",     "Generic relation from co-occurrence extraction",
         None, None),
        (ONTO.affiliatedWith,"affiliatedWith", "Person affiliated with an organization",
         ONTO.Person, ONTO.Organization),
        (ONTO.worksAt,       "worksAt",        "Person works at an organization",
         ONTO.Person, ONTO.Organization),
        (ONTO.authorOf,      "authorOf",       "Person is author of a research paper",
         ONTO.Person, ONTO.ResearchPaper),
        (ONTO.locatedIn,     "locatedIn",      "Organization located in a geopolitical entity",
         ONTO.Organization, ONTO.GeopoliticalEntity),
        (ONTO.receivedAward, "receivedAward",  "Person received an award",
         ONTO.Person, ONTO.Award),
        (ONTO.educatedAt,    "educatedAt",     "Person was educated at an organization",
         ONTO.Person, ONTO.Organization),
    ]

    for prop_uri, label, comment, domain, range_ in obj_props:
        g.add((prop_uri, RDF.type,       OWL.ObjectProperty))
        g.add((prop_uri, RDFS.label,     Literal(label, lang="en")))
        g.add((prop_uri, RDFS.comment,   Literal(comment, lang="en")))
        if domain:
            g.add((prop_uri, RDFS.domain, domain))
        if range_:
            g.add((prop_uri, RDFS.range, range_))

    # Datatype property 
    g.add((ONTO.extractedFrom, RDF.type,     OWL.DatatypeProperty))
    g.add((ONTO.extractedFrom, RDFS.label,   Literal("extractedFrom", lang="en")))
    g.add((ONTO.extractedFrom, RDFS.comment, Literal("Source URL of the web page from which the entity was extracted", lang="en")))
    g.add((ONTO.extractedFrom, RDFS.range,   XSD.anyURI))

    return g


# Consolidated alignment 
def consolidate_alignment() -> Graph:
    """Merges the entity + predicate alignment files."""
    g = Graph()
    g.bind("onto",  ONTO)
    g.bind("wd",    WD)
    g.bind("wdt",   WDT)
    g.bind("owl",   OWL)
    g.bind("rdfs",  RDFS)

    for align_file in ["alignment_entities.ttl", "alignment_predicates.ttl"]:
        path = OUTPUT_DIR / align_file
        if path.exists():
            g.parse(str(path), format="turtle")
            print(f"  ✓ Loaded: {align_file}")
        else:
            print(f"  ⚠️  Not found: {align_file}")

    return g


# Statistics report
def generate_stats_report(private_stats, expanded_stats, split_stats, align_count):
    # Helper
    def fmt(d: dict, key: str, default="N/A"):
        v = d.get(key, default)
        return f"{v:,}" if isinstance(v, int) else str(v)

    report = f"""
{'='*60}
  STATISTICS REPORT — AI RESEARCH KNOWLEDGE BASE
{'='*60}
  Domain  : AI Research (LLMs, Machine Learning, Deep Learning)
  Source  : Web pages from distill.pub + AI publications
{'='*60}

1. INITIAL KB (private_kb.nt)
{'─'*40}
  Triples            : {fmt(private_stats, 'n_triples')}
  Entities           : {fmt(private_stats, 'n_entities')}
  Unique relations   : {fmt(private_stats, 'n_relations')}
  Distinct subjects  : {fmt(private_stats, 'n_subjects')}

  Type distribution:"""

    for type_label, count in (private_stats.get("type_distribution") or []):
        report += f"\n    {type_label:30s} : {count:>6,}"

    report += f"""

2. FINAL EXPANDED KB (expanded_kb.nt)
{'─'*40}
  Triples            : {fmt(expanded_stats, 'n_triples')}
  Entities           : {fmt(expanded_stats, 'n_entities')}
  Unique relations   : {fmt(expanded_stats, 'n_relations')}

  Top 10 predicates:"""

    for pred_uri, count in (expanded_stats.get("top_predicates") or []):
        pred_short = pred_uri.split("/")[-1][:40]
        report += f"\n    {pred_short:40s} : {count:>8,}"

    total_split = sum(split_stats.values())
    report += f"""

3. KGE SPLIT
{'─'*40}
  Train              : {split_stats.get('train', 0):>10,}  ({split_stats.get('train',0)/max(total_split,1)*100:.1f}%)
  Validation         : {split_stats.get('valid', 0):>10,}  ({split_stats.get('valid',0)/max(total_split,1)*100:.1f}%)
  Test               : {split_stats.get('test',  0):>10,}  ({split_stats.get('test',0)/max(total_split,1)*100:.1f}%)
  Total              : {total_split:>10,}

4. ALIGNMENTS
{'─'*40}
  Alignment triples  : {align_count:>6,}
  (owl:sameAs + owl:equivalentProperty + rdfs:subPropertyOf)

5. COMPLIANCE WITH PROJECT CONSTRAINTS
{'─'*40}
  Initial KB ≥ 100 triples    : {'✓' if private_stats.get('n_triples',0) >= 100 else '✗'}  ({fmt(private_stats,'n_triples')} triples)
  Initial KB ≥ 50 entities    : {'✓' if private_stats.get('n_entities',0) >= 50 else '✗'}  ({fmt(private_stats,'n_entities')} entities)
  Final KB ≥ 50,000 triples   : {'✓' if expanded_stats.get('n_triples',0) >= 50_000 else '✗'}  ({fmt(expanded_stats,'n_triples')} triples)
  Final KB ≤ 200,000 triples  : {'✓' if expanded_stats.get('n_triples',0) <= 200_000 else '⚠️ Exceeded'}
  Entities 5k-30k             : {'✓' if 5000 <= expanded_stats.get('n_entities',0) <= 30000 else '✗'}  ({fmt(expanded_stats,'n_entities')} entities)
  Relations 50-200            : {'✓' if 50 <= expanded_stats.get('n_relations',0) <= 200 else '~'}  ({fmt(expanded_stats,'n_relations')} relations)

{'='*60}
"""
    return report


# Deliverables check 
def check_deliverables():
    required = [
        OUTPUT_DIR / "private_kb.nt",
        OUTPUT_DIR / "expanded_kb.nt",
        OUTPUT_DIR / "ontology.ttl",
        OUTPUT_DIR / "alignment_full.ttl",
        OUTPUT_DIR / "stats_report.txt",
        KGE_DIR    / "train.txt",
        KGE_DIR    / "valid.txt",
        KGE_DIR    / "test.txt",
        KGE_DIR    / "entities.dict",
        KGE_DIR    / "relations.dict",
    ]
    print("\n[STEP 6] Checking deliverables:")
    all_ok = True
    for path in required:
        exists = path.exists()
        size   = path.stat().st_size if exists else 0
        status = f"✓ {size:>10,} bytes" if exists else "✗ MISSING"
        print(f"  {status}  {path}")
        if not exists:
            all_ok = False

    return all_ok


#  Main 
def run_generate_deliverables():
    print("\n[STEP 6] Generating final deliverables...")

    # Statistics
    print("\n  Computing statistics...")
    private_stats  = compute_stats(OUTPUT_DIR / "private_kb.nt")
    expanded_stats = compute_stats(OUTPUT_DIR / "expanded_kb.nt")
    split_stats    = compute_split_stats()

    # Ontology
    print("\n  Generating ontology...")
    onto_graph = generate_ontology()
    onto_path  = OUTPUT_DIR / "ontology.ttl"
    onto_graph.serialize(destination=str(onto_path), format="turtle")
    print(f"  ✓ {onto_path} ({len(onto_graph)} triples)")

    # Consolidated alignment
    print("\n  Consolidating alignments...")
    align_graph = consolidate_alignment()
    align_path  = OUTPUT_DIR / "alignment_full.ttl"
    align_graph.serialize(destination=str(align_path), format="turtle")
    align_count = len(align_graph)
    print(f"  ✓ {align_path} ({align_count} triples)")

    # Report
    report = generate_stats_report(private_stats, expanded_stats, split_stats, align_count)
    report_path = OUTPUT_DIR / "stats_report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)
    print(report)
    print(f"  → {report_path}")

    # Final check
    all_ok = check_deliverables()

    if all_ok:
        print("\n✅  All deliverables are present. KB is ready for KGE!")
    else:
        print("\n⚠️   Some deliverables are missing. Run the previous steps.")


if __name__ == "__main__":
    run_generate_deliverables()


[STEP 6] Generating final deliverables...

  Computing statistics...

  Generating ontology...
  ✓ output\ontology.ttl (63 triples)

  Consolidating alignments...
  ✓ Loaded: alignment_entities.ttl
  ✓ Loaded: alignment_predicates.ttl
  ✓ output\alignment_full.ttl (313 triples)

  STATISTICS REPORT — AI RESEARCH KNOWLEDGE BASE
  Domain  : AI Research (LLMs, Machine Learning, Deep Learning)
  Source  : Web pages from distill.pub + AI publications

1. INITIAL KB (private_kb.nt)
────────────────────────────────────────
  Triples            : 2,246
  Entities           : 434
  Unique relations   : 8
  Distinct subjects  : 426

  Type distribution:
    Person                         :    297
    Organization                   :     74
    Date                           :     21
    Entity                         :     20
    GeopoliticalEntity             :      6
    owl#Class                      :      5
    owl#ObjectProperty             :      3
    owl#DatatypeProperty           :   